# 07-units seeds (Bogota, PM2.5) -- stability of the constrained search

The PM2.5 lead equation (complexity 18 of `bogota_pm25_pysr_units_pareto_front.csv`) came from one unseeded run. This notebook repeats exactly that search (`07_units_equation_discovery.ipynb`: same pool, units, operators, `niterations=200`, `populations=24`, `maxsize=25`) with three fixed seeds, so every constrained search in the paper has the same stability check as ozone. It does not overwrite the lead front.

Scoring (same protocol as the Bogota lead equations, notebooks A2/A4): leave-one-year-out, constants refitted by non-linear least squares on the training years, **both** pooled R2 and mean of per-year R2, on the rows the search used. The existing free-form front is scored on the same rows.

In [1]:
# %% Settings (edit here, then Run All)
CITY, TARGET, TAG = 'bogota', 'pm25', 'units_seeds'
MASTER = 'bogota_master_dataset.parquet'                         # file in Data/processed/colombia/<city>/
SELECTION_JSON = 'bogota_feature_selection.json'                    # its selected_no_lag is the pool (identical to the free-form run)
EXTRA_FEATURES = []                  # added to that pool (none added)
DROP_FEATURES = []                    # removed from it (split into raw factors)
SEEDS = [0, 1, 2]                           # symbolic search is stochastic: report all of them
PYSR_KW = dict(niterations=200, binary_operators=["+", "-", "*", "/"], unary_operators=["log", "sqrt", "exp"],
               model_selection="best", maxsize=25, populations=24)     # identical to the Bogota constrained runs
UNIT_MAP = {'w700': 'Pa/s', 'co': 'kg/m^3', 'hotspots_sector_E_lag3': '', 'tcwv_lag1': 'kg/m^2', 'precip_total_mm_lag1': 'mm', 'wind_speed_era5': 'm/s', 'radiation_mean': 'W/m^2', 'hotspots_sector_N_lag2': ''}
Y_UNITS = 'kg/m^3'
FREEFORM_FRONT = 'bogota_pm25_pysr_units_pareto_front.csv'                     # the existing free-form front, scored here on the same rows
MAX_COMPLEXITY_REPORTED = 25

In [2]:
import sys, json, re, warnings
from pathlib import Path
PROJECT_ROOT = Path("/path/to/project")
sys.path.insert(0, str(PROJECT_ROOT / "Scripts"))
import numpy as np, pandas as pd, sympy as sp
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score
DATA_DIR = PROJECT_ROOT / "Data" / "processed" / "colombia" / CITY
df = pd.read_parquet(DATA_DIR / MASTER)
df["date"] = pd.to_datetime(df["date"]); df["year"] = df["date"].dt.year
df = df.dropna(subset=[TARGET]).reset_index(drop=True)
selection = json.load(open(DATA_DIR / SELECTION_JSON))
features = [f for f in selection["selected_no_lag"] if f not in DROP_FEATURES]
for f in EXTRA_FEATURES:
    if f not in features: features.append(f)
print("pool:", features)
data = df.dropna(subset=features + [TARGET]).reset_index(drop=True)
X = data[features].to_numpy(); y = data[TARGET].to_numpy()
X_units = [UNIT_MAP[f] for f in features]
print(data.shape, dict(zip(features, X_units)), "years:", sorted(data.year.unique()))

pool: ['w700', 'co', 'hotspots_sector_E_lag3', 'tcwv_lag1', 'precip_total_mm_lag1', 'wind_speed_era5', 'radiation_mean', 'hotspots_sector_N_lag2']
(3282, 216) {'w700': 'Pa/s', 'co': 'kg/m^3', 'hotspots_sector_E_lag3': '', 'tcwv_lag1': 'kg/m^2', 'precip_total_mm_lag1': 'mm', 'wind_speed_era5': 'm/s', 'radiation_mean': 'W/m^2', 'hotspots_sector_N_lag2': ''} years: [np.int32(2014), np.int32(2015), np.int32(2016), np.int32(2017), np.int32(2018), np.int32(2019), np.int32(2020), np.int32(2021), np.int32(2022)]


In [3]:
from pysr import PySRRegressor
fronts = {}
for seed in SEEDS:
    sr = PySRRegressor(**PYSR_KW, random_state=seed)
    sr.fit(X, y, variable_names=features, X_units=X_units, y_units=Y_UNITS)
    f = sr.equations_[["complexity", "loss", "equation"]].copy(); f["seed"] = seed
    f.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_pareto_front_seed{seed}.csv", index=False)
    fronts[seed] = f
    print("seed", seed, "done; best:", sr.get_best()["equation"])

Compiling Julia backend...


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(


┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:



Expressions evaluated per second: 2.050e+04
Progress: 50 / 200 total iterations (25.438%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.444e+01  0.000e+00  y[m⁻³ kg] = 18.297[?]
3           3.988e+01  2.399e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.84[?]
5           3.984e+01  4.807e-04  y[m⁻³ kg] = (co[m⁻³ kg] * 23.437[?]) - -0.4606[?]
6           3.366e+01  1.686e-01  y[m⁻³ kg] = sqrt((co[m⁻³ kg] / wind_speed_era5[m s...
                                              ⁻¹]) * 431.34[?])
7           3.164e+01  6.189e-02  y[m⁻³ kg] = (23.266[?] - (w700[s⁻¹ Pa] * 28.144[?]...
                                              )) * co[m⁻³ kg]
8           3.000e+01  5.324e-02  y[m⁻³ kg] = sqrt(((co[m⁻³ kg] * radiation_mean[m⁻²...
                                               W]) * 2.29[?]) / win

seed 0 done; best: sqrt((427.12177 / wind_speed_era5) * co)


/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73



Expressions evaluated per second: 2.190e+04
Progress: 53 / 200 total iterations (26.812%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.444e+01  0.000e+00  y[m⁻³ kg] = 18.293[?]
3           3.988e+01  2.399e-01  y[m⁻³ kg] = co[m⁻³ kg] / 0.041877[?]
5           3.986e+01  3.350e-04  y[m⁻³ kg] = (co[m⁻³ kg] - -0.076618[?]) / 0.045689...
                                              [?]
6           3.364e+01  1.695e-01  y[m⁻³ kg] = sqrt(co[m⁻³ kg] / wind_speed_era5[m s⁻...
                                              ¹]) * 20.617[?]
7           3.205e+01  4.846e-02  y[m⁻³ kg] = (co[m⁻³ kg] * 23.933[?]) - (w700[s⁻¹ P...
                                              a] * 17.713[?])
8           3.128e+01  2.445e-02  y[m⁻³ kg] = sqrt(co[m⁻³ kg]) * (21.864[?] - (w700[...
            

[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:
/path/to/project/.venv/lib/python3.11/site-packages/pysr/sr.py:2385: UserWarning: Note: Setting `random_state` without also setting `deterministic=True` and `parallelism='serial'` will result in non-deterministic searches.
  warnings.warn(
┌ Warning: You are using dimensional constraints, but `dimensional_constraint_penalty` was not set. The default penalty of `1000.0` will be used.
└ @ SymbolicRegression ~/.julia/packages/SymbolicRegression/O7o63/src/Configure.jl:73
[ Info: Started!
[ Info: Final population:
[ Info: Results saved to:


Expressions evaluated per second: 2.100e+04
Progress: 51 / 200 total iterations (25.729%)
════════════════════════════════════════════════════════════════════════════════════════════════════
───────────────────────────────────────────────────────────────────────────────────────────────────
Complexity  Loss       Score      Equation
1           6.444e+01  0.000e+00  y[m⁻³ kg] = 18.293[?]
3           3.988e+01  2.399e-01  y[m⁻³ kg] = co[m⁻³ kg] * 23.868[?]
5           3.982e+01  7.698e-04  y[m⁻³ kg] = 0.65034[?] - (co[m⁻³ kg] * -23.09[?])
6           3.364e+01  1.686e-01  y[m⁻³ kg] = 20.615[?] * sqrt(co[m⁻³ kg] / wind_spe...
                                              ed_era5[m s⁻¹])
7           3.205e+01  4.855e-02  y[m⁻³ kg] = (co[m⁻³ kg] + (w700[s⁻¹ Pa] / -1.1525[...
                                              ?])) * 24.109[?]
8           3.165e+01  1.240e-02  y[m⁻³ kg] = co[m⁻³ kg] * (exp(w700[s⁻¹ Pa] * -1.45...
                                              95[?]) * 22.829[?])
9 

In [4]:
FLOAT = re.compile(r"(?<![A-Za-z_0-9.])\d+\.\d+(?:[eE][+-]?\d+)?")
def make_model(eq, feats):
    vals = []
    def sub(m): vals.append(float(m.group(0))); return f"c{len(vals)-1}"
    s = FLOAT.sub(sub, eq)
    syms = {f: sp.Symbol(f) for f in feats}; cs = [sp.Symbol(f"c{i}") for i in range(len(vals))]
    expr = sp.sympify(s, locals={**syms, **{str(c): c for c in cs}, "sqrt": sp.sqrt, "log": sp.log, "exp": sp.exp})
    used = [f for f in feats if syms[f] in expr.free_symbols]
    keep = [i for i, c in enumerate(cs) if c in expr.free_symbols]
    fn = sp.lambdify([syms[f] for f in used] + [cs[i] for i in keep], expr, "numpy")
    p0 = np.array([vals[i] for i in keep])
    return (lambda Xt, *p: fn(*Xt, *p)), used, p0

yy = data[TARGET].to_numpy(float); years = data["year"].to_numpy(); YRS = sorted(set(years))
def score(eq, feats):
    model, used, p0 = make_model(eq, feats)
    if not used:                                   # constant equation: leave-one-year-out mean
        oof = np.concatenate([np.full((years == yr).sum(), np.mean(yy[years != yr])) for yr in YRS]); yo = np.concatenate([yy[years == yr] for yr in YRS])
        per = [r2_score(yy[years == yr], np.full((years == yr).sum(), np.mean(yy[years != yr]))) for yr in YRS]
        return dict(r2_insample=0.0, r2_pooled=r2_score(yo, oof), r2_mean_years=float(np.mean(per)), vars="")
    Xt = tuple(data[f].to_numpy(float) for f in used)
    with np.errstate(all="ignore"), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        try: popt, _ = curve_fit(model, Xt, yy, p0=p0, maxfev=20000)
        except Exception: popt = p0
        pred = np.full(len(yy), np.nan)
        for yr in YRS:
            te = years == yr
            try: pf, _ = curve_fit(model, tuple(a[~te] for a in Xt), yy[~te], p0=popt, maxfev=20000)
            except Exception: pf = popt
            pred[te] = model(tuple(a[te] for a in Xt), *pf)
        ok = np.all(np.isfinite(pred)); ins = model(Xt, *popt)
        per = [r2_score(yy[years == yr], pred[years == yr]) for yr in YRS] if ok else [np.nan]
        return dict(r2_insample=r2_score(yy, ins) if np.all(np.isfinite(ins)) else np.nan, r2_pooled=r2_score(yy, pred) if ok else np.nan,
                    r2_mean_years=float(np.mean(per)), vars=",".join(used))

rows = []
for seed, f in fronts.items():
    for _, r in f.iterrows():
        if r.complexity <= MAX_COMPLEXITY_REPORTED: rows.append(dict(seed=str(seed), complexity=int(r.complexity), equation=r.equation, **score(r.equation, features)))
ff = pd.read_csv(DATA_DIR / FREEFORM_FRONT).sort_values("complexity")
ff_feats = sorted({t for e in ff.equation for t in re.findall(r"[A-Za-z_][A-Za-z0-9_]*", e) if t in data.columns})
for _, r in ff.iterrows():
    if r.complexity <= MAX_COMPLEXITY_REPORTED:
        try: rows.append(dict(seed="free-form", complexity=int(r.complexity), equation=r.equation, **score(r.equation, ff_feats)))
        except Exception as e: print("free-form eq skipped", r.complexity, e)
scored = pd.DataFrame(rows); scored.to_csv(DATA_DIR / f"{CITY}_{TARGET}_pysr_{TAG}_scored.csv", index=False)
pd.set_option("display.width", 250, "display.max_colwidth", 110)
print("n days", len(data)); print(scored[["seed", "complexity", "r2_pooled", "r2_mean_years", "vars"]].round(3).to_string(index=False))

n days 3282
     seed  complexity  r2_pooled  r2_mean_years                                                                  vars
        0           1     -0.017         -0.085                                                                      
        0           3      0.370          0.341                                                                    co
        0           5      0.368          0.337                                                                    co
        0           6      0.465          0.439                                                    co,wind_speed_era5
        0           7      0.497          0.477                                                               w700,co
        0           8      0.522          0.502                                     co,wind_speed_era5,radiation_mean
        0          10      0.528          0.508                                     co,wind_speed_era5,radiation_mean
        0          11      0.532          0.